In [3]:
import pandas as pd
import altair as alt

# Load your data -- replace with your actual path
df = pd.read_csv("results.csv")
param = 'avg_GPUTime'
factor = 'screen_pct'

# Average over all parameter values within each (scene, resolution)
# since parameters don't affect GPU time
df_agg = (
    df.groupby(['scene', factor])[param]
    .mean()
    .reset_index()
)

# Mean across scenes per resolution
df_mean = (
    df_agg.groupby(factor)[param]
    .mean()
    .reset_index()
    .assign(scene='Mean')
)

df_plot = pd.concat([df_agg, df_mean], ignore_index=True)

# Individual scene lines
scene_lines = alt.Chart(df_plot[df_plot['scene'] != 'Mean']).mark_line(
    point=True, opacity=0.4, strokeWidth=1
).encode(
    x=alt.X(f'{factor}:O', title='Resolution (%)'),
    y=alt.Y(f'{param}:Q', title='Mean GPU Time (ms)'),
    color=alt.Color('scene:N', legend=alt.Legend(title='Scene')),
)

# Bold mean line
mean_line = alt.Chart(df_plot[df_plot['scene'] == 'Mean']).mark_line(
    point=True, strokeWidth=3, color='black'
).encode(
    x=alt.X(f'{factor}:O', title='Resolution (%)'),
    y=alt.Y(f'{param}:Q'),
)

chart = (scene_lines + mean_line).properties(
    title='Total GPU Time vs. Rendering Resolution',
    width=1000, height=500
)

chart.show()
chart.save('gputime.png')

alt.LayerChart(...)

In [13]:
# For each (scene, resolution, parameter), compute std of avg_GPUTime across parameter values
# This tells us: how much does GPU time vary as we change each parameter?

df_param_std = (
    df.groupby(['scene', 'screen_pct', 'cvar_name'])['avg_GPU/TAA']
    .std()
    .reset_index()
    .rename(columns={'avg_GPU/TAA': 'std_across_values'})
)

# Then average that std across scenes, per (parameter, resolution)
df_summary = (
    df_param_std.groupby(['cvar_name', 'screen_pct'])['std_across_values']
    .agg(mean_std='mean', max_std='max')
    .reset_index()
)

print(df_summary.to_string(index=False))

                           cvar_name  screen_pct  mean_std  max_std
r.TemporalAA.HistoryScreenPercentage          50  0.000468 0.001848
r.TemporalAA.HistoryScreenPercentage          71  0.000348 0.001050
r.TemporalAA.HistoryScreenPercentage          87  0.000377 0.000794
r.TemporalAA.HistoryScreenPercentage         100  0.000411 0.000695
      r.TemporalAACurrentFrameWeight          50  0.000635 0.002572
      r.TemporalAACurrentFrameWeight          71  0.000326 0.001046
      r.TemporalAACurrentFrameWeight          87  0.000362 0.000616
      r.TemporalAACurrentFrameWeight         100  0.000508 0.001548
              r.TemporalAAFilterSize          50  0.000472 0.001084
              r.TemporalAAFilterSize          71  0.000281 0.000757
              r.TemporalAAFilterSize          87  0.000406 0.000593
              r.TemporalAAFilterSize         100  0.000383 0.001095
                 r.TemporalAASamples          50  0.000461 0.001521
                 r.TemporalAASamples          71

In [4]:
import pandas as pd
import altair as alt

df = pd.read_csv("results.csv")

param = 'avg_GPU/TAA'
param_title = 'Mean GPU/TAA Time (ms)'

cvars = df['cvar_name'].unique()

charts = []

for cvar in cvars:
    df_cvar = df[df['cvar_name'] == cvar].copy()

    # Average over scenes for each cvar_value
    df_agg = (
        df_cvar.groupby(['scene', 'cvar_value'])[param]
        .mean()
        .reset_index()
    )

    # Mean across scenes per cvar_value
    df_mean = (
        df_agg.groupby('cvar_value')[param]
        .mean()
        .reset_index()
        .assign(scene='Mean')
    )

    df_plot = pd.concat([df_agg, df_mean], ignore_index=True)

    scene_lines = alt.Chart(df_plot[df_plot['scene'] != 'Mean']).mark_line(
        point=True, opacity=0.4, strokeWidth=1
    ).encode(
        x=alt.X('cvar_value:O', title=f'{cvar} value'),
        y=alt.Y(f'{param}:Q', title=param_title),
        color=alt.Color('scene:N', legend=alt.Legend(title='Scene')),
    )

    mean_line = alt.Chart(df_plot[df_plot['scene'] == 'Mean']).mark_line(
        point=True, strokeWidth=3, color='black'
    ).encode(
        x=alt.X('cvar_value:O'),
        y=alt.Y(f'{param}:Q'),
    )

    chart = (scene_lines + mean_line).properties(
        title=f'{cvar}',
        width=400, height=300
    )

    charts.append(chart)

# Arrange in a 2x2 grid
grid = alt.vconcat(
    alt.hconcat(*charts[:2]),
    alt.hconcat(*charts[2:])
).properties(
    title=f'{param_title} per CVar'
)

grid.show()
grid.save('taa-time.png')

alt.VConcatChart(...)

In [35]:
import pandas as pd
import altair as alt

RESOLUTION_LABELS = {50: '25%', 71: '50%', 87: '75%', 100: '100%'}

res_order  = ['100%', '75%', '50%', '25%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

df = pd.read_csv("results.csv")
df['screen_pct_label'] = df['screen_pct'].astype(int).map(RESOLUTION_LABELS)

def plot_cvar_performance(df, param, param_title=None):
    if param_title is None:
        param_title = param

    cvars = df['cvar_name'].unique()
    charts = []

    for cvar in cvars:
        df_cvar = df[df['cvar_name'] == cvar]

        df_agg = (
            df_cvar.groupby(['scene', 'cvar_value', 'screen_pct_label'])[param]
            .mean()
            .reset_index()
        )

        df_mean = (
            df_agg.groupby(['cvar_value', 'screen_pct_label'])[param]
            .mean()
            .reset_index()
        )

        chart = alt.Chart(df_mean).mark_line(point=True).encode(
            x=alt.X('cvar_value:O', title=f'{cvar} value'),
            y=alt.Y(f'{param}:Q', title=param_title),
            color=alt.Color('screen_pct_label:N',
                            title='Screen %',
                            scale=color_scale,
                            sort=res_order),
        ).properties(
            title=f'{cvar}',
            width=400, height=300
        )
        charts.append(chart)

    n = len(charts)
    ncols = 2
    rows = [charts[i:i+ncols] for i in range(0, n, ncols)]

    grid = alt.vconcat(
        *[alt.hconcat(*row) for row in rows]
    ).resolve_scale(
        color='shared'
    ).properties(
        title=f'{param_title} per CVar'
    )

    return grid


grid = plot_cvar_performance(df, param='avg_GPUMem/LocalUsedMB', param_title='Mean avg_GPUMem/LocalUsedMB Time (ms)')
grid.show()

alt.VConcatChart(...)